# broadcasting-rules — ex1: predict the broadcast shape

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcasting-rules`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five broadcasting patterns that ramp from predicting the result shape → row-vector broadcast → column-vector broadcast → targeted axis insertion → outer product via broadcast. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Vectorization and broadcasting` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `broadcasting-rules`**, which bridges to the bank subtopic `Numpy: Vectorization and broadcasting` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcasting-rules"
DD_SUBTOPIC = "Numpy: Vectorization and broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Broadcasting — quick refresher

**The rule** (NumPy & PyTorch agree):
1. Right-align both shapes; left-pad the shorter with 1s.
2. For each pair of aligned axes: equal → keep; one is 1 → use the other; otherwise → incompatible.

**Three patterns you reach for constantly:**
- **Row broadcast** — `(N, D) + (D,)` works automatically. Adds a per-feature bias.
- **Column broadcast** — `(N, D) * w` where `w` is `(N,)` fails. Reshape `w` to `(N, 1)` first.
- **Axis insertion** — `unsqueeze` / `[:, None]` / `reshape` are all valid ways to insert a size-1 axis where broadcasting needs it.

### Exercise 1 — predict the broadcast shape

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall and apply the right-align broadcasting rule on two shape tuples.
> Keywords: shape-rule, right-align, incompatibility
> ```

**KCs targeted:** `predict-broadcast-shape`

Implement `ex1_broadcast_shape(shape_a, shape_b)` to return the shape that would result from broadcasting two tensors of the given shapes, OR raise `ValueError` if they're incompatible.

**The rule** (NumPy & PyTorch agree):
1. Right-align both shapes, left-pad the shorter with 1s.
2. For each pair `(a, b)` of aligned axes: keep if equal; if exactly one is 1, use the other; otherwise → incompatible.

Return the result as a tuple of ints.

**Examples:**
- `(3, 4)` and `(4,)` → `(3, 4)`
- `(2, 1, 3)` and `(5, 3)` → `(2, 5, 3)`
- `(3, 4)` and `(3,)` → `ValueError` (right-align mismatch on last axis)

In [ ]:
def ex1_broadcast_shape(shape_a, shape_b):
    """Return broadcasted shape (tuple) or raise ValueError if incompatible."""
    raise NotImplementedError()


def _test_ex1():
    # Compatible cases
    assert ex1_broadcast_shape((3, 4), (4,)) == (3, 4)
    assert ex1_broadcast_shape((3, 4), (1, 4)) == (3, 4)
    assert ex1_broadcast_shape((3, 1), (4,)) == (3, 4)
    assert ex1_broadcast_shape((2, 1, 3), (5, 3)) == (2, 5, 3)
    assert ex1_broadcast_shape((1,), (5, 6, 7)) == (5, 6, 7)
    assert ex1_broadcast_shape((), (5,)) == (5,)

    # Incompatible cases — must raise ValueError
    raised = False
    try:
        ex1_broadcast_shape((3, 4), (3,))   # right-align: (3,4) vs (1,3) → mismatch at axis -1
    except ValueError:
        raised = True
    assert raised, 'should have raised ValueError for incompatible shapes (3,4) vs (3,)'

    raised = False
    try:
        ex1_broadcast_shape((2, 3), (4, 3))
    except ValueError:
        raised = True
    assert raised, 'should have raised ValueError for (2,3) vs (4,3)'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_broadcast_shape(shape_a, shape_b):
    a = list(shape_a)
    b = list(shape_b)
    n = max(len(a), len(b))
    a = [1] * (n - len(a)) + a
    b = [1] * (n - len(b)) + b
    out = []
    for ai, bi in zip(a, b):
        if ai == bi:
            out.append(ai)
        elif ai == 1:
            out.append(bi)
        elif bi == 1:
            out.append(ai)
        else:
            raise ValueError(f'incompatible axes: {ai} vs {bi}')
    return tuple(out)
```

**Why right-align?** Trailing axes correspond to the fastest-varying memory layout. Aligning shapes from the right means a `(D,)` vector broadcasts across rows of an `(N, D)` matrix, which is the natural 'one weight per feature' case. Left-align would have given you 'one weight per sample' instead — a different, much less common need.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()